In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.metrics import classification_report, accuracy_score

# 1. LOAD DATASET
# Replace 'RT_IOT2022.csv' with your actual file path
df = pd.read_csv('RT_IOT2022.csv')

# 2. PREPROCESSING
# Identify categorical columns and encode them
categorical_cols = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])



In [ ]:
# Define Features and Target
# Using the top features relevant to IoT traffic
X = df.drop(['Attack_type'], axis=1) 
y = df['Attack_type']

# Identify the 'Normal' label index for the detection logic later
# Usually, LabelEncoder assigns 0, 1, 2... alphabetically. 
# We will find the numeric value for 'Normal' traffic.
labels_map = dict(zip(le.classes_, range(len(le.classes_))))
normal_label_value = labels_map.get('Normal', 0) 

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. TRAIN MODELS
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

print("Training SVM...")
svm_model = svm.SVC(kernel='rbf', probability=True)
svm_model.fit(X_train_scaled, y_train)

# 4. DETECTION SYSTEM FUNCTION
def detect_iot_attack(network_flow_data, model):
    """
    Takes raw network flow data, scales it, and predicts if it's an attack.
    """
    # Ensure the data is in the correct shape (1 row, N features)
    scaled_data = scaler.transform(network_flow_data.values.reshape(1, -1))
    
    # Predict
    prediction = model.predict(scaled_data)[0]
    probabilities = model.predict_proba(scaled_data)[0]
    confidence = np.max(probabilities) * 100
    
    if prediction == normal_label_value:
        return f"✅ Status: Normal Traffic (Confidence: {confidence:.2f}%)"
    else:
        # Reverse the label encoding to show the attack name
        attack_name = le.inverse_transform([prediction])[0]
        return f"⚠️ ALERT: {attack_name} Detected! (Confidence: {confidence:.2f}%)"

# 5. SIMULATING DETECTION
print("\n--- Starting Detection Simulator ---")

# Pick a random sample from the test set to simulate a 'new' incoming flow
sample_index = np.random.randint(0, len(X_test))
new_flow = X_test.iloc[sample_index]
actual_label = le.inverse_transform([y_test.iloc[sample_index]])[0]

print(f"Simulating flow for actual type: {actual_label}")

# Run Detection
print("RF Result:", detect_iot_attack(new_flow, rf_model))
print("SVM Result:", detect_iot_attack(new_flow, svm_model))

Training Random Forest...
Training SVM...

--- Starting Detection Simulator ---
Simulating flow for actual type: DOS_SYN_Hping


C:\Users\ADITI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


RF Result: ⚠️ ALERT: DOS_SYN_Hping Detected! (Confidence: 100.00%)
SVM Result: ⚠️ ALERT: DOS_SYN_Hping Detected! (Confidence: 100.00%)


C:\Users\ADITI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [4]:
print(df['Attack_type'].nunique())


12
